# T35 - 不同曲线投资顺序分析
## 第5章：回测分析

### 概述
本章进行策略回测和收益计算，包括：
1. 加载债券收益率数据和策略配置
2. 实现债券收益计算公式
3. 对每个周期进行策略回测
4. 计算策略的年化收益率
5. 汇总回测结果

In [ ]:
# 1. 导入必要的库
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import json
import warnings

warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("库导入成功！")

In [ ]:
# 2. 加载配置
from config import (
    TYPE_MAPPING,
    RATING_MAPPING,
    DURATION_MAP,
    INITIAL_AMOUNT,
    OUTPUT_DIR,
    YIELD_DATA_FILE,
    CYCLE_LABEL_FILE
)

print(f"初始投资金额: {INITIAL_AMOUNT:,.0f} 元")

In [ ]:
# 3. 加载数据

def load_data():
    """加载所需数据"""
    print("加载数据...")
    print("=" * 60)
    
    # 加载收益率数据
    if os.path.exists(YIELD_DATA_FILE):
        yield_data = pd.read_csv(YIELD_DATA_FILE, encoding='utf-8-sig')
        yield_data['dt'] = pd.to_datetime(yield_data['dt'])
        print(f"收益率数据: {len(yield_data)} 条记录")
    else:
        raise FileNotFoundError(f"收益率数据文件不存在: {YIELD_DATA_FILE}")
    
    # 加载周期标注
    if os.path.exists(CYCLE_LABEL_FILE):
        cycles_df = pd.read_csv(CYCLE_LABEL_FILE, encoding='utf-8-sig')
        cycles_df['start_date'] = pd.to_datetime(cycles_df['start_date'])
        cycles_df['end_date'] = pd.to_datetime(cycles_df['end_date'])
        print(f"周期标注: {len(cycles_df)} 个周期")
    else:
        raise FileNotFoundError(f"周期标注文件不存在: {CYCLE_LABEL_FILE}")
    
    # 加载策略配置
    strategy_config_file = os.path.join(OUTPUT_DIR, '策略配置.json')
    if os.path.exists(strategy_config_file):
        with open(strategy_config_file, 'r', encoding='utf-8') as f:
            strategy_config = json.load(f)
        print(f"策略配置: {len(strategy_config['dimensions'])} 个维度")
    else:
        raise FileNotFoundError(f"策略配置文件不存在: {strategy_config_file}")
    
    return yield_data, cycles_df, strategy_config

yield_data, cycles_df, strategy_config = load_data()

In [ ]:
# 4. 定义Strategy类（从上一章复制）

class Strategy:
    """债券投资策略类"""
    
    def __init__(self, bond_type, term, credit_level=None):
        self.bond_type = bond_type
        self.term = term
        self.credit_level = credit_level
    
    def __str__(self):
        if self.credit_level:
            return f"{self.bond_type}_{self.term}_{self.credit_level}资质"
        return f"{self.bond_type}_{self.term}"
    
    def __repr__(self):
        return self.__str__()
    
    def get_duration(self):
        """获取修正久期"""
        return DURATION_MAP.get(self.term, 4.0)


def split_period(start_date, end_date, n_splits):
    """将时间段等分为n个子期间"""
    if isinstance(start_date, str):
        start_date = pd.to_datetime(start_date)
    if isinstance(end_date, str):
        end_date = pd.to_datetime(end_date)
    
    total_days = (end_date - start_date).days
    days_per_split = total_days // n_splits
    
    periods = []
    for i in range(n_splits):
        if i == n_splits - 1:
            period_end = end_date
        else:
            period_end = start_date + pd.Timedelta(days=(i + 1) * days_per_split)
        
        period_start = start_date + pd.Timedelta(days=i * days_per_split)
        periods.append((period_start, period_end))
    
    return periods

print("Strategy类和辅助函数已定义")

In [ ]:
# 5. 定义债券收益计算函数

def calculate_bond_return(start_yield, end_yield, duration, period_days):
    """
    计算债券在持有期间的总收益率
    
    债券总收益 = 票息收益 + 资本利得/损失
    
    Parameters:
    -----------
    start_yield : float
        期初收益率（小数形式，如0.035表示3.5%）
    end_yield : float
        期末收益率（小数形式）
    duration : float
        修正久期（年）
    period_days : int
        持有天数
        
    Returns:
    --------
    float
        总收益率（小数形式）
    
    Notes:
    -------
    票息收益 = 期初收益率 × (持有天数 / 365)
    价格收益 = -修正久期 × 收益率变动 × (持有天数 / 365)
    总收益 = 票息收益 + 价格收益
    """
    # 1. 票息收益（使用期初收益率作为票息率）
    coupon_return = start_yield * (period_days / 365)
    
    # 2. 资本利得/损失
    # 收益率下降，债券价格上升，资本利得为正
    yield_change = end_yield - start_yield
    price_return = -duration * yield_change * (period_days / 365)
    
    # 3. 总收益
    total_return = coupon_return + price_return
    
    return total_return

# 测试收益计算函数
print("测试债券收益计算函数")
print("=" * 60)

# 示例：期初收益率3.5%，期末3.0%，久期5年，持有180天
test_return = calculate_bond_return(0.035, 0.030, 5, 180)
print(f"\n示例:")
print(f"  期初收益率: 3.5%")
print(f"  期末收益率: 3.0%")
print(f"  修正久期: 5年")
print(f"  持有天数: 180天")
print(f"\n计算过程:")
print(f"  票息收益 = 3.5% × (180/365) = {0.035 * (180/365):.4f} = {0.035 * (180/365)*100:.2f}%")
print(f"  收益率变动 = 3.0% - 3.5% = -0.5%")
print(f"  价格收益 = -5 × (-0.5%) × (180/365) = {-5 * (-0.005) * (180/365):.4f} = {-5 * (-0.005) * (180/365)*100:.2f}%")
print(f"  总收益 = {test_return:.4f} = {test_return*100:.2f}%")

In [ ]:
# 6. 定义回测引擎类

class BacktestEngine:
    """
    回测引擎类
    
    用于执行策略回测和收益计算
    """
    
    def __init__(self, yield_data, cycles_df, initial_amount=INITIAL_AMOUNT):
        """
        初始化回测引擎
        
        Parameters:
        -----------
        yield_data : DataFrame
            债券收益率数据
        cycles_df : DataFrame
            周期标注数据
        initial_amount : float
            初始投资金额
        """
        self.yield_data = yield_data
        self.cycles_df = cycles_df
        self.initial_amount = initial_amount
        self.type_mapping = TYPE_MAPPING
        self.rating_mapping = RATING_MAPPING
    
    def get_period_yields(self, strategy, start_date, end_date):
        """
        获取特定策略在特定时期的收益率序列
        
        Parameters:
        -----------
        strategy : Strategy
            投资策略
        start_date : datetime
            起始日期
        end_date : datetime
            结束日期
            
        Returns:
        --------
        Series
            收益率序列
        """
        # 筛选日期范围
        period_data = self.yield_data[
            (self.yield_data['dt'] >= start_date) & 
            (self.yield_data['dt'] <= end_date)
        ].copy()
        
        # 筛选债券类型
        bond_types = self.type_mapping.get(strategy.bond_type, [strategy.bond_type])
        period_data = period_data[period_data['bond_type'].isin(bond_types)]
        
        # 筛选期限
        period_data = period_data[period_data['term'] == strategy.term]
        
        # 筛选评级
        if strategy.credit_level and strategy.bond_type != '利率债':
            ratings = self.rating_mapping.get(strategy.credit_level, [])
            period_data = period_data[period_data['rating'].isin(ratings)]
        
        if period_data.empty:
            return pd.Series()
        
        # 按日期计算平均收益率
        daily_yields = period_data.groupby('dt')['yield_rate'].mean()
        
        return daily_yields
    
    def run_strategy_sequence(self, strategy_sequence, cycle_id):
        """
        在特定周期内运行策略序列
        
        Parameters:
        -----------
        strategy_sequence : list
            策略序列，每个元素是(bond_type, term, credit_level)元组
        cycle_id : int
            周期ID
            
        Returns:
        --------
        dict
            回测结果
        """
        # 获取周期信息
        cycle_info = self.cycles_df[self.cycles_df['cycle'] == cycle_id].iloc[0]
        start_date = cycle_info['start_date']
        end_date = cycle_info['end_date']
        
        # 分割周期
        periods = split_period(start_date, end_date, len(strategy_sequence))
        
        # 执行策略序列
        portfolio_value = self.initial_amount
        returns_sequence = []
        
        for i, ((period_start, period_end), (bond_type, term, credit_level)) in enumerate(
            zip(periods, strategy_sequence)
        ):
            strategy = Strategy(bond_type, term, credit_level)
            
            # 获取收益率
            yields = self.get_period_yields(strategy, period_start, period_end)
            
            if not yields.empty:
                start_yield = yields.iloc[0] / 100
                end_yield = yields.iloc[-1] / 100
                duration = strategy.get_duration()
                period_days = (period_end - period_start).days
                
                # 计算收益
                period_return = calculate_bond_return(start_yield, end_yield, duration, period_days)
                portfolio_value *= (1 + period_return)
                returns_sequence.append(period_return)
            else:
                returns_sequence.append(0)
        
        # 计算整体收益率
        total_return = (portfolio_value - self.initial_amount) / self.initial_amount
        cycle_days = (end_date - start_date).days
        annual_return = total_return * (365 / cycle_days) if cycle_days > 0 else 0
        
        return {
            'cycle_id': cycle_id,
            'strategy_sequence': ' -> '.join([str(Strategy(*s)) for s in strategy_sequence]),
            'total_return': total_return,
            'annual_return': annual_return,
            'returns_sequence': returns_sequence,
            'final_value': portfolio_value
        }
    
    def run_backtest(self, all_strategies):
        """
        运行所有策略的回测
        
        Parameters:
        -----------
        all_strategies : dict
            按维度分类的策略组合
            
        Returns:
        --------
        dict
            按维度分类的回测结果
        """
        results = {}
        
        for dimension, strategy_list in all_strategies.items():
            print(f"\n回测 {dimension} 策略...")
            dimension_results = []
            
            for cycle_id in self.cycles_df['cycle'].unique():
                for strategy_seq in strategy_list:
                    result = self.run_strategy_sequence(strategy_seq, cycle_id)
                    dimension_results.append(result)
            
            results[dimension] = pd.DataFrame(dimension_results)
            print(f"  完成 {len(dimension_results)} 条回测记录")
        
        return results

# 创建回测引擎
engine = BacktestEngine(yield_data, cycles_df)
print("BacktestEngine 类已定义")

In [ ]:
# 7. 从配置重建策略

def rebuild_strategies_from_config(strategy_config):
    """从配置重建策略"""
    all_strategies = {}
    
    for dimension, config in strategy_config['dimensions'].items():
        strategy_list = []
        for strategy in config['strategies']:
            seq = [(step['bond_type'], step['term'], step['credit_level']) for step in strategy['steps']]
            strategy_list.append(seq)
        all_strategies[dimension] = strategy_list
    
    return all_strategies

all_strategies = rebuild_strategies_from_config(strategy_config)

print("从配置重建策略")
print("=" * 60)
for dimension, strategies in all_strategies.items():
    print(f"{dimension}: {len(strategies)} 个策略序列")

In [ ]:
# 8. 运行回测

print("开始运行回测...")
print("=" * 60)

backtest_results = engine.run_backtest(all_strategies)

print("\n回测完成！")

In [ ]:
# 9. 显示回测结果

def display_backtest_results(results):
    """显示回测结果"""
    print("回测结果")
    print("=" * 60)
    
    for dimension, df in results.items():
        print(f"\n{dimension}:")
        print(f"  回测记录数: {len(df)}")
        
        # 按策略序列汇总
        summary = df.groupby('strategy_sequence').agg({
            'annual_return': ['mean', 'std', 'min', 'max', 'count']
        }).round(4)
        summary.columns = ['平均年化收益率', '收益率标准差', '最小收益率', '最大收益率', '周期数']
        
        print("\n  策略统计:")
        print(summary.to_string())
        
        # 找出最优策略
        best_strategy = summary['平均年化收益率'].idxmax()
        best_return = summary.loc[best_strategy, '平均年化收益率']
        print(f"\n  最优策略: {best_strategy}")
        print(f"  平均年化收益率: {best_return:.2%}")

display_backtest_results(backtest_results)

In [ ]:
# 10. 保存回测结果

def save_backtest_results(results):
    """保存回测结果"""
    print("保存回测结果...")
    print("=" * 60)
    
    for dimension, df in results.items():
        # 保存详细结果
        output_file = os.path.join(OUTPUT_DIR, f'{dimension}回测结果.csv')
        df.to_csv(output_file, index=False, encoding='utf-8-sig', float_format='%.4f')
        print(f"{dimension}回测结果已保存: {output_file}")
        
        # 保存策略统计
        stats = df.groupby('strategy_sequence').agg({
            'annual_return': ['mean', 'std', 'min', 'max'],
            'total_return': ['mean', 'std', 'min', 'max']
        }).round(4)
        stats.columns = [
            '年化收益率均值', '年化收益率标准差', '年化收益率最小值', '年化收益率最大值',
            '总收益率均值', '总收益率标准差', '总收益率最小值', '总收益率最大值'
        ]
        
        stats_file = os.path.join(OUTPUT_DIR, f'{dimension}策略统计.csv')
        stats.to_csv(stats_file, encoding='utf-8-sig')
        print(f"{dimension}策略统计已保存: {stats_file}")

save_backtest_results(backtest_results)

In [ ]:
# 11. 各周期详细分析

def analyze_by_cycle(results):
    """按周期分析策略表现"""
    print("各周期策略表现分析")
    print("=" * 60)
    
    for dimension, df in results.items():
        print(f"\n{dimension}:")
        
        # 按周期和策略汇总
        cycle_analysis = df.pivot_table(
            index='cycle_id',
            columns='strategy_sequence',
            values='annual_return',
            aggfunc='mean'
        ).round(4)
        
        print("\n  各周期年化收益率:")
        print(cycle_analysis.to_string())
        
        # 找出每个周期的最优策略
        print("\n  各周期最优策略:")
        for cycle_id in cycle_analysis.index:
            best_strategy = cycle_analysis.loc[cycle_id].idxmax()
            best_return = cycle_analysis.loc[cycle_id].max()
            print(f"    周期 {cycle_id}: {best_strategy} ({best_return:.2%})")

analyze_by_cycle(backtest_results)

### 总结

本章完成了以下工作：
1. 加载了债券收益率数据、周期标注和策略配置
2. 定义了债券收益计算公式（票息收益 + 资本利得）
3. 实现了回测引擎类 `BacktestEngine`
4. 对三个维度的策略进行了完整回测
5. 计算了每个策略的年化收益率
6. 保存了回测结果和策略统计

**收益计算公式:**
- 票息收益 = 期初收益率 × (持有天数 / 365)
- 价格收益 = -修正久期 × 收益率变动 × (持有天数 / 365)
- 总收益 = 票息收益 + 价格收益

**输出文件:**
- `品种维度回测结果.csv`
- `品种维度策略统计.csv`
- `久期维度回测结果.csv`
- `久期维度策略统计.csv`
- `资质维度回测结果.csv`
- `资质维度策略统计.csv`

**下一步**: 运行 `06-可视化.ipynb` 生成分析图表。